# Animation: RL Agent Learning to Navigate

Watch the agent go from **total chaos** to **optimal navigation** as Q-Learning trains it.

- **Early episodes:** random walk, usually falls into holes
- **Middle episodes:** starting to find the goal sometimes
- **Late episodes:** consistently takes the shortest safe path

No external libraries — pure `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(42)

# ── Grid environment (same as Ch10/Ch11) ────────────────────────
#  3=Start  0=Free  1=Hole  2=Goal
GRID = np.array([[3,0,0,0],[0,1,0,1],[0,0,0,1],[1,0,0,2]])
ROWS, COLS = 4, 4
N_STATES, N_ACTIONS = 16, 4
MOVE_DR = {0:(0,-1), 1:(1,0), 2:(0,1), 3:(-1,0)}
REWARDS = {0:-0.01, 1:-1.0, 2:1.0, 3:-0.01}

_state = [0]
def env_reset():
    _state[0] = 0
    return 0

def env_step(action):
    r, c = _state[0]//COLS, _state[0]%COLS
    dr, dc = MOVE_DR[action]
    nr = max(0, min(ROWS-1, r+dr))
    nc = max(0, min(COLS-1, c+dc))
    ns = nr*COLS + nc
    ct = GRID[nr, nc]
    _state[0] = ns
    return ns, REWARDS[ct], ct in [1, 2]

# ── Q-Learning ──────────────────────────────────────────────────
ALPHA, GAMMA = 0.8, 0.95
N_EPISODES, MAX_STEPS = 3000, 100

Q = np.zeros((N_STATES, N_ACTIONS))
epsilon = 1.0

# Save snapshots of agent trajectories at specific training points
SNAPSHOT_EPISODES = [0, 50, 200, 500, 1000, 2999]
snapshots = {}  # episode → trajectory

def get_trajectory(Q, epsilon=0.0, max_steps=MAX_STEPS):
    state = env_reset()
    traj = [(state//COLS, state%COLS)]
    for _ in range(max_steps):
        if np.random.random() < epsilon:
            action = np.random.randint(N_ACTIONS)
        else:
            action = np.argmax(Q[state])
        next_state, reward, done = env_step(action)
        traj.append((next_state//COLS, next_state%COLS))
        state = next_state
        if done:
            break
    return traj

for episode in range(N_EPISODES):
    if episode in SNAPSHOT_EPISODES:
        snap_eps = min(epsilon, 0.8) if episode < 100 else 0.1 if episode < 500 else 0.0
        snapshots[episode] = get_trajectory(Q.copy(), epsilon=snap_eps)

    state = env_reset()
    for _ in range(MAX_STEPS):
        action = np.random.randint(N_ACTIONS) if np.random.random() < epsilon else np.argmax(Q[state])
        ns, reward, done = env_step(action)
        future = 0 if done else GAMMA * np.max(Q[ns])
        Q[state, action] += ALPHA * (reward + future - Q[state, action])
        state = ns
        if done:
            break
    epsilon = max(epsilon * 0.995, 0.01)

# Add final snapshot after training
snapshots[2999] = get_trajectory(Q, epsilon=0.0)

print('Training complete!')
print('Trajectory lengths per snapshot:', {k: len(v) for k, v in snapshots.items()})

In [ ]:
# ── Helper: draw the grid ────────────────────────────────────────
CELL_COLORS = {0:'#ecf0f1', 1:'#e74c3c', 2:'#2ecc71', 3:'#3498db'}
CELL_LABELS = {0:'F', 1:'H', 2:'G', 3:'S'}

def draw_grid(ax):
    for r in range(ROWS):
        for c in range(COLS):
            ct = GRID[r, c]
            rect = mpatches.FancyBboxPatch(
                [c, ROWS-1-r], 1, 1,
                boxstyle='round,pad=0.05',
                facecolor=CELL_COLORS[ct], edgecolor='white', linewidth=3
            )
            ax.add_patch(rect)
            ax.text(c+0.5, ROWS-1-r+0.5, CELL_LABELS[ct],
                    ha='center', va='center', fontsize=16,
                    fontweight='bold', color='#2c3e50')
    ax.set_xlim(0, COLS)
    ax.set_ylim(0, ROWS)
    ax.set_aspect('equal')
    ax.axis('off')

In [ ]:
# ── Animation 1: Agent trajectory at each training snapshot ─────
snap_keys = sorted(snapshots.keys())
ep_labels = {0:'Episode 0\n(untrained)', 50:'Episode 50', 200:'Episode 200',
             500:'Episode 500', 1000:'Episode 1000', 2999:'Episode 3000\n(trained)'}

fig, ax = plt.subplots(figsize=(5.5, 6))
draw_grid(ax)

trail_line, = ax.plot([], [], '-', color='#2c3e50', lw=2.5, alpha=0.6, zorder=4)
agent_dot,  = ax.plot([], [], 'o', color='#f39c12', ms=18, zorder=5,
                      markeredgecolor='#2c3e50', markeredgewidth=1.5)
title_obj   = ax.set_title('', fontsize=13, pad=12)
step_text   = ax.text(0.5, -0.04, '', transform=ax.transAxes,
                      fontsize=10, ha='center')

# We'll animate: for each snapshot, play out the trajectory step by step
# Build a flat frame list: (traj, ep_label, step_in_traj)
frames = []
HOLD_FRAMES = 18  # freeze at end of each trajectory
for ep in snap_keys:
    traj = snapshots[ep]
    label = ep_labels[ep]
    for step in range(len(traj)):
        frames.append((traj, label, step))
    for _ in range(HOLD_FRAMES):
        frames.append((traj, label, len(traj)-1))

def update(frame_idx):
    traj, label, step = frames[frame_idx]
    pts = traj[:step+1]

    # Convert (row, col) to plot coords
    xs = [c + 0.5 for r, c in pts]
    ys = [ROWS - r - 0.5 for r, c in pts]

    trail_line.set_data(xs[:-1], ys[:-1])
    agent_dot.set_data([xs[-1]], [ys[-1]])

    r, c = traj[step]
    ct = GRID[r, c]
    outcome = ''
    if ct == 2:
        outcome = ' ✓ GOAL!'
    elif ct == 1:
        outcome = ' ✗ hole'

    title_obj.set_text(label)
    step_text.set_text(f'Step {step}{outcome}')
    return trail_line, agent_dot, title_obj, step_text

anim = FuncAnimation(fig, update, frames=len(frames),
                     interval=220, repeat=True, blit=True)
plt.tight_layout()
plt.close()
HTML(anim.to_jshtml())

In [ ]:
# ── Bonus: learned policy heatmap ───────────────────────────────
action_arrows = {0:'←', 1:'↓', 2:'→', 3:'↑'}
best_actions  = np.argmax(Q, axis=1)
special = {5:'H', 7:'H', 11:'H', 12:'H', 15:'G', 0:'S'}

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Policy grid
ax = axes[0]
draw_grid(ax)
for s in range(N_STATES):
    r, c = s//COLS, s%COLS
    label = special.get(s, action_arrows[best_actions[s]])
    color = '#e74c3c' if s in [5,7,11,12] else '#2ecc71' if s==15 else '#3498db' if s==0 else '#2c3e50'
    ax.text(c+0.5, ROWS-1-r+0.15, label, ha='center', va='center',
            fontsize=22, fontweight='bold', color=color, zorder=6)
ax.set_title('Learned Policy\n(arrows = best action)', fontsize=12)

# Q-value heatmap (max Q per state)
ax2 = axes[1]
max_q = Q.max(axis=1).reshape(ROWS, COLS)
im = ax2.imshow(max_q, cmap='YlGn', aspect='equal')
plt.colorbar(im, ax=ax2, label='Max Q-value')
for r in range(ROWS):
    for c in range(COLS):
        s = r*COLS + c
        label = special.get(s, f'{max_q[r,c]:.2f}')
        ax2.text(c, r, label, ha='center', va='center', fontsize=12,
                 fontweight='bold', color='#2c3e50')
ax2.set_title('Max Q-value per State\n(brighter = more valuable)', fontsize=12)
ax2.set_xticks([])
ax2.set_yticks([])

plt.suptitle('Q-Learning: What the Agent Learned', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## What to notice

- **Episode 0:** The agent wanders randomly — it has no knowledge yet
- **Episode 50–200:** The agent starts to find the goal occasionally, but path is still erratic
- **Episode 1000+:** The agent reliably follows the shortest safe route

The Q-value heatmap shows the **value** of each cell: states close to the goal have high Q-values because the agent learned that being there leads to reward.

The policy arrows show the **best action** the agent would take from each cell — this *is* the optimal policy the agent learned entirely from trial and error.